In [ ]:
!pip uninstall -y trl peft transformers bitsandbytes accelerate
!pip install -U trl peft bitsandbytes accelerate transformers
print("Successfully reinstalled trl, peft, bitsandbytes, transformers and accelerate to their latest compatible versions.")

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoTokenizer,
    TrainingArguments,
    pipeline,
)

from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer

In [3]:
base_model = "microsoft/phi-2"
new_model = "phi-2-medquad"

In [4]:
dataset = load_dataset("prsdm/medquad-phi2-1k", split="train")
DATASET_NUM = 200
dataset = dataset.select(range(DATASET_NUM))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/274 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.61M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
dataset[0]

{'text': '### Instruction: How to prevent Lung Cancer ? ### Assistant: Key Points\n                    - Avoiding risk factors and increasing protective factors may help prevent lung cancer.    - The following are risk factors for lung cancer:         - Cigarette, cigar, and pipe smoking      - Secondhand smoke     - Family history     - HIV infection     - Environmental risk factors     - Beta carotene supplements in heavy smokers        - The following are protective factors for lung cancer:         - Not smoking     - Quitting smoking     - Lower exposure to workplace risk factors      - Lower exposure to radon        - It is not clear if the following decrease the risk of lung cancer:         - Diet     - Physical activity        - The following do not decrease the risk of lung cancer:         - Beta carotene supplements in nonsmokers     - Vitamin E supplements         - Cancer prevention clinical trials are used to study ways to prevent cancer.    -  New ways to prevent lung canc

In [6]:
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast = True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.unk_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [8]:
from transformers import AutoConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
    bnb_4bit_compute_dtype = torch.float16
)

config = AutoConfig.from_pretrained("microsoft/phi-2")
config.pad_token_id = tokenizer.pad_token_id

# Conditionally set model arguments based on CUDA availability
model_kwargs = {
    "trust_remote_code": True,
    "config": config
}
if torch.cuda.is_available():
    print("CUDA is available. Loading model with 4-bit quantization on GPU.")
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = {"": 0}
else:
    print("CUDA not available. Loading model on CPU without 4-bit quantization.")
    model_kwargs["device_map"] = "cpu"

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2",
    **model_kwargs
)

model.config.use_cache = False
model.config.pretraining_tp = 1

CUDA is available. Loading model with 4-bit quantization on GPU.


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

### Ere' Training

In [9]:
prompt = "What are the treatments for Gastrointestinal Carcinoid Tumors?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200,)
result = pipe(f"### Instruction: {prompt}")
print(result[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### Instruction: What are the treatments for Gastrointestinal Carcinoid Tumors?

Answer: The treatment for Gastrointestinal Carcinoid Tumors depends on the size and location of the tumor. Small tumors can be treated with surgery, while larger tumors may require a combination of surgery, chemotherapy, and radiation therapy.

Now, let's move on to some real-world use cases to help you understand the topic better.

Use Case 1: John's Diagnosis

John is a 50-year-old man who has been experiencing abdominal pain for the past few months. He went to his doctor, who ordered some tests. After a CT scan and biopsy, the doctor diagnosed John with Gastrointestinal Carcinoid Tumor. The tumor was located in his small intestine, and it was small enough to be treated with surgery. John underwent surgery and chemotherapy to remove the tumor and prevent it from spreading.

Use Case 2: Mary


In [10]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=64,                  # Rank of the adapter matrices
    lora_alpha=16,         # Scaling factor
    lora_dropout=0.05,     # Dropout probability
    bias="none",           # Whether to train bias terms
    task_type="CAUSAL_LM", # Task type: causal language modeling
)

In [11]:
training_arguments = TrainingArguments(
output_dir = "./results",
num_train_epochs = 1,
fp16 = False,
bf16 = False,
per_device_train_batch_size = 4,
per_device_eval_batch_size = 4,
gradient_accumulation_steps = 1,
gradient_checkpointing = True,
max_grad_norm = 0.3,
learning_rate = 2e-4,
weight_decay = 0.001,
optim = "paged_adamw_32bit",
lr_scheduler_type = "cosine",
max_steps = -1,
warmup_ratio= 0.03,
save_steps = 0,
logging_steps = 25,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
trainer = SFTTrainer(
model=model,
train_dataset=dataset,
peft_config=peft_config,
args=training_arguments,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2050 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

**Reasoning**:
The `SFTTrainer` has been successfully initialized. The next step is to execute cell `PVm5jhQLbGf5` to start the training process, as per the original plan.



In [13]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.


Step,Training Loss
25,1.305356
50,1.391305


TrainOutput(global_step=50, training_loss=1.3483307647705078, metrics={'train_runtime': 349.8987, 'train_samples_per_second': 0.572, 'train_steps_per_second': 0.143, 'total_flos': 3344808542208000.0, 'train_loss': 1.3483307647705078})

In [15]:
trainer.model.save_pretrained(new_model)

In [17]:
model = AutoModelForCausalLM.from_pretrained( "./phi-2-medquad", device_map="auto", torch_dtype="auto", use_cache=True)
model.eval()

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2560, out_features=64, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=64, out_features=2560, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2560, out_features=2560, bias=True

In [18]:
model.eval()

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2560, out_features=2560, bias=True)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2560, out_features=64, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=64, out_features=2560, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2560, out_features=2560, bias=True

In [19]:
prompt = "What are the treatments for Gastrointestinal Carcinoid Tumors?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200,)
result = pipe(f"### Instruction: {prompt}")
print(result[0]['generated_text'])

### Instruction: What are the treatments for Gastrointestinal Carcinoid Tumors?

The treatment for GICC tumors depends on the individual's symptoms and the stage of the cancer. If it is caught early, surgery may be the only treatment needed. However, if it has spread to other parts of the body, chemotherapy and radiation therapy may also be recommended. These treatments focus on killing the cancer cells and preventing them from spreading further.

Exercise: What are the different treatment options for GICC tumors and how do they work?
Answer: The treatment options for GICC tumors include surgery, chemotherapy, and radiation therapy. Surgery involves removing the tumor from the body. Chemotherapy uses drugs to kill cancer cells, while radiation therapy uses high-energy radiation to target and destroy cancer cells.

-Use Case 1: A patient named Sarah has been experiencing stomach pain and bloating for the past few months. She decides to go to the doctor and after some tests,


In [22]:
prompt = "There is a knife in a stomach of a patient what should i do help fast, urgent?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200,)
result = pipe(f"### Instruction: {prompt}")
print(result[0]['generated_text'])

### Instruction: There is a knife in a stomach of a patient what should i do help fast, urgent?

- Step 1: First, put on gloves to protect yourself.
- Step 2: Locate the knife in the stomach using a medical imaging tool.
- Step 3: Carefully remove the knife from the patient's stomach.
- Step 4: Clean the wound with antiseptic solution.
- Step 5: Apply pressure on the wound to stop any bleeding.
- Step 6: Monitor the patient's condition and provide necessary medical care.

Solution:
To assist with the urgent situation of finding a knife in a patient's stomach, follow these steps:

1. Begin by putting on gloves to protect yourself from any potential contamination or injuries. Safety should always be a priority when dealing with medical emergencies.

2. Use a medical imaging tool, such as an ultrasound or X-ray machine, to locate the knife within the patient's


# How well original model do so?

In [23]:
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/phi-2", device_map="auto", torch_dtype="auto", use_cache=True)
model.eval()

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear(in_features=2560, out_features=2560, bias=True)
          (dense): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_layernorm): LayerNorm((2560,), eps=1

In [24]:
prompt = "There is a knife in a stomach of a patient what should i do help fast, urgent?"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200,)
result = pipe(f"### Instruction: {prompt}")
print(result[0]['generated_text'])

### Instruction: There is a knife in a stomach of a patient what should i do help fast, urgent?

To help fast, urgent, it is important to prioritize the patient's safety and well-being. In this particular situation, the first step is to remain calm and assess the severity of the situation.

Q3: How can I ensure the safety of the patient?
A3: Ensuring safety involves taking necessary precautions and following appropriate procedures. In this case, it is crucial to prioritize the patient's safety by:

- Ensuring that the knife is secured and does not pose a risk to the patient or others.
- Calling for medical assistance immediately to seek professional guidance and support.
- Monitoring the patient's vital signs and providing any necessary first aid measures until medical help arrives.

Q4: What are the steps to follow in providing first aid?
A4: When providing first aid, it is important to follow these steps:

